# Обучение категоризатора трат (Попытка 1, до поиска датасета Дрёмова)

Это третий обучаемый компонент системы. Он берёт извлечённые из чека
магазин и список позиций и относит трату к одной из шести категорий: продукты,
кафе и рестораны, транспорт, аптека, развлечения, прочее.

В отличие от детектора и Donut, у этой задачи нет готового публичного датасета
под наши категории и под русский язык. Поэтому мы обучаемся на синтетическом
наборе, который собирается из справочников реальных российских магазинов и
типичных товаров по каждой категории. Это решает сразу три проблемы, которые мы
заранее видели: дисбаланс классов (баланс задаём вручную), язык (данные русские,
а не малайско-английские, как в SROIE и CORD) и воспроизводимость (один seed
порождает один и тот же датасет).

Вход модели - не одно название магазина, а короткая текстовая сигнатура вида
«магазин. товар, товар, ...». Это вытаскивает часть чеков из «Прочего».

На инференсе категоризатор работает в связке с правилами: известные сети
классифицируются мгновенно по словарю, а модель берёт на себя всё, что правила
не покрыли. Здесь же мы обучаем именно модельную часть и меряем её
качество на тесте.

## Запуск в Google Colab

Ноутбук я специально подготовила для удобного запуска в Colab без ручных скачиваний.
Код и данные подтягиваются из удалённых источников автоматически.

In [1]:
# Настройка окружения для Colab
import os
import sys
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
# Публичный git-репозиторий с кодом проекта
REPO_URL = "https://github.com/ScarletFlame611/Receipt-AI.git"


def _find_project_root():
    here = Path.cwd()
    for cand in [here, *here.parents]:
        if (cand / "src").is_dir():
            return cand
    for sub in sorted(p for p in here.iterdir() if p.is_dir()):
        if (sub / "src").is_dir():
            return sub
    return None


if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "transformers", "datasets", "scikit-learn"],
        check=False,
    )
    if _find_project_root() is None and "USER/" not in REPO_URL:
        subprocess.run(["git", "clone", REPO_URL], check=True)
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

root = _find_project_root()
if root is None:
    raise RuntimeError(
        "Не найден код проекта (папка src/). В Colab склонируй репозиторий "
        "(или задай REPO_URL) и перезапусти ячейку."
    )
os.chdir(root)
sys.path.insert(0, str(root))
print("Colab:", IN_COLAB, "| корень проекта:", root)

Colab: True | корень проекта: /content/Receipt-AI


## Настройка окружения

Проверяем GPU и ставим зависимости. Датасет категоризатора синтетический и
генерируется прямо здесь кодом проекта (детерминированно, фиксированный seed):
три файла train.jsonl, validation.jsonl и test.jsonl, в каждой строке объект
с текстовой сигнатурой «магазин. товары» и меткой категории.

In [2]:
import torch
print("GPU доступен:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Устройство:", torch.cuda.get_device_name(0))

GPU доступен: True
Устройство: Tesla T4


In [3]:
!pip install transformers datasets scikit-learn -q

In [4]:
from pathlib import Path
from src.data.datasets import save_categorizer_dataset

# Датасет категоризатора синтетический
data_root = root / "data" / "processed" / "categorizer_data"
if not (data_root / "train.jsonl").exists():
    save_categorizer_dataset(data_root, per_class=500, seed=42)
print("папка данных существует:", data_root.exists())
for name in ["train", "validation", "test"]:
    p = data_root / f"{name}.jsonl"
    if p.exists():
        n = len(p.read_text(encoding="utf-8").strip().splitlines())
        print(f"{name}: {n} примеров")
    else:
        print(f"{name}: нет файла")


папка данных существует: True
train: 2400 примеров
validation: 300 примеров
test: 300 примеров


GPU подключён, датасет на месте: 2400 примеров на обучение,
300 на валидацию и 300 на тест. Размеры совпадают с тем, что заложено при
генерации (по 500 на каждую из шести категорий, разбивка 80/10/10).

## Загрузка и осмотр данных

Читаем три части из jsonl и смотрим, что именно подаётся модели на вход. Каждый
пример - это пара: текстовая сигнатура и метка категории. Полезно убедиться
своими глазами, что сигнатуры выглядят как реальный чек после извлечения
(магазин плюс несколько позиций) и что классы в обучающей части сбалансированы,
иначе модель будет перекошена в сторону частого класса. Заодно фиксируем список
категорий, он должен совпадать с порядком меток из конфига проекта.

In [5]:
import json
from collections import Counter

def load_jsonl(path):
    return [json.loads(l) for l in path.read_text(encoding="utf-8").strip().splitlines()]

train_rows = load_jsonl(data_root / "train.jsonl")
val_rows = load_jsonl(data_root / "validation.jsonl")
test_rows = load_jsonl(data_root / "test.jsonl")
# Порядок категорий
labels = ["Продукты", "Кафе и рестораны", "Транспорт", "Аптека", "Развлечения", "Прочее"]
print("категории:", labels)

print("\nпримеры из train:")
for r in train_rows[:8]:
    print(f"  [{r['label']}]  {r['text']}")

print("\nбаланс классов в train:")
for label, n in Counter(r["label"] for r in train_rows).most_common():
    print(f"  {label}: {n}")

категории: ['Продукты', 'Кафе и рестораны', 'Транспорт', 'Аптека', 'Развлечения', 'Прочее']

примеры из train:
  [Аптека]  Фармаимпекс. пластырь, мазь заживляющая, парацетамол, йод, цитрамон
  [Продукты]  Глобус. помидоры, батон, картофель, соль, яйца, масло сливочное
  [Транспорт]  Тройка. омыватель стекла, билет на метро, заправка полный бак
  [Транспорт]  Аэроэкспресс. билет РЖД, бензин АИ-98, билет электричка
  [Аптека]  Мелодия Здоровья. мазь заживляющая, сироп от кашля, супрастин
  [Развлечения]  Боулинг Сити. квест комната, билет на концерт, VR сеанс, тир
  [Прочее]  Фикс Прайс. удлинитель, полотенце, розетка, наушники, пакеты для мусора
  [Продукты]  Верный. яйца, молоко, бананы, творог, сосиски

баланс классов в train:
  Аптека: 414
  Прочее: 405
  Транспорт: 402
  Кафе и рестораны: 396
  Продукты: 393
  Развлечения: 390


Сигнатуры выглядят как реальный чек после извлечения: магазин плюс несколько
позиций через запятую. Заодно заметна заложенная при генерации зашумлённость. Это намеренно чтобы модель не привыкала, что все
позиции в чеке строго одной категории, и училась опираться на совокупность
сигналов, а не на одно слово.

Баланс классов почти идеальный: от 390 до 414 примеров на категорию. Перекоса,
который мог бы увести модель в сторону частого класса, нет.

## Токенизация и подготовка к обучению

Модель не работает с текстом напрямую, сигнатуру нужно разбить на токены и
перевести в числа. Берём токенизатор от bert-base-multilingual-cased: эта версия
BERT обучена на ста с лишним языках, включая русский, поэтому понимает наши
названия магазинов и товаров без отдельной настройки. Метки категорий тоже
переводим в числа: каждой категории сопоставляем индекс по фиксированному
порядку из списка labels, чтобы модель и мы говорили об одних и тех же классах
одинаково.

Сигнатуры короткие: магазин и несколько позиций, поэтому ограничение длины в
128 токенов покрывает их с большим запасом и ничего не обрезает. Оборачиваем
данные в формат datasets и прогоняем токенизацию пакетами.

In [6]:
from transformers import AutoTokenizer
from datasets import Dataset

base_model = "bert-base-multilingual-cased"
max_length = 128
tokenizer = AutoTokenizer.from_pretrained(base_model)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

def to_dataset(rows):
    return Dataset.from_dict({
        "text": [r["text"] for r in rows],
        "labels": [label2id[r["label"]] for r in rows],
    })

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=max_length)

train_ds = to_dataset(train_rows).map(tokenize, batched=True, remove_columns=["text"])
val_ds = to_dataset(val_rows).map(tokenize, batched=True, remove_columns=["text"])
test_ds = to_dataset(test_rows).map(tokenize, batched=True, remove_columns=["text"])

lengths = [len(tokenizer(r["text"])["input_ids"]) for r in train_rows]
print("длина в токенах - медиана:", sorted(lengths)[len(lengths)//2], "максимум:", max(lengths))
print("label2id:", label2id)
print(train_ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/2400 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

длина в токенах - медиана: 27 максимум: 63
label2id: {'Продукты': 0, 'Кафе и рестораны': 1, 'Транспорт': 2, 'Аптека': 3, 'Развлечения': 4, 'Прочее': 5}
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2400
})


Токенизация отработала на всех трёх частях. Длина сигнатур в токенах невелика:
медиана 27, максимум 63, это заметно ниже лимита в 128, поэтому ни один чек не
обрезается, и при этом мы не тратим память. Метки переведены
в индексы по фиксированному порядку категорий, и в датасете появились все поля,
которые ждёт модель: input_ids, attention_mask и token_type_ids плюс labels.
Данные полностью готовы к обучению.

## Модель и настройка обучения

Загружаем bert-base-multilingual-cased с классификационной головой на шесть
классов. Голова инициализируется случайно и это нормально, именно её и нижние
слои мы и дообучаем под нашу задачу. В модель передаём соответствие меток и
индексов, чтобы потом на инференсе она возвращала человекочитаемые названия
категорий, а не голые номера.

Метрику для отбора лучшей модели берём macro-F1, а не обычную точность т.к. macro-F1 усредняет F1 по всем категориям с равным весом, поэтому
если модель завалит редкий или трудный класс, метрика это покажет, а accuracy
могла бы замаскировать провал за счёт лёгких классов.

Параметры обучения: небольшая скорость обучения, потому что
дообучаем уже готовую модель, десять эпох, отбор лучшей версии по валидации.
Лучшую модель тренер сам подхватит в конце по максимуму macro-F1.

In [7]:
import numpy as np
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    base_model,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label,
)

def compute_metrics(eval_pred):
    logits, gold = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(gold, preds),
        "macro_f1": f1_score(gold, preds, average="macro"),
    }

ckpt_dir = str(root / "runs" / "categorizer")

args = TrainingArguments(
    output_dir=ckpt_dir,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=25,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
print("готово к обучению")

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


готово к обучению


## Запуск обучения

Запускаем дообучение на десять эпох. После каждой эпохи тренер прогоняет
валидацию и показывает потери на обучении, потери на валидации, accuracy и
macro-F1. Чекпойнты сохраняются локально, а в конце тренер сам подгрузит лучшую
по macro-F1 версию.

In [8]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.012838,0.026698,0.996667,0.996724
2,0.003585,0.017678,0.996667,0.996724
3,0.002422,0.016383,0.996667,0.996724
4,0.001387,0.020586,0.996667,0.996724
5,0.001046,0.021130,0.996667,0.996724
6,0.000851,0.021708,0.996667,0.996724
7,0.000726,0.022492,0.996667,0.996724
8,0.000663,0.022435,0.996667,0.996724
9,0.000622,0.022208,0.996667,0.996724
10,0.000605,0.022367,0.996667,0.996724


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1500, training_loss=0.047449166973431905, metrics={'train_runtime': 844.8646, 'train_samples_per_second': 28.407, 'train_steps_per_second': 1.775, 'total_flos': 550530321266304.0, 'train_loss': 0.047449166973431905, 'epoch': 10.0})

Обучение сошлось быстро: уже после первой эпохи macro-F1 близок к
единице, валидационные потери в целом снижаются, лучшая модель по валидации
зафиксирована на десятой эпохе с accuracy и macro-F1 около 1.0.

Но к этим резцльтатом нужно относиться осторожно. Почти идеальное качество объясняется
природой данных: датасет синтетический, и обучающая, валидационная и тестовая
части собраны из одних и тех же справочников магазинов и товаров. Модель по сути
выучила эту лексику, а проверяется на новых сочетаниях знакомых слов, поэтому
ошибаться ей почти негде.

## Оценка на тесте

Прогоняем лучшую модель на тестовой части, которую не видели при обучении, и
печатаем разбивку по каждой категории, а не только усреднённую метрику. Отчёт
показывает precision, recall и F1 отдельно для каждого класса. Это важнее общего
числа: если бы какая-то категория проседала, например аптека путалась бы с
продуктами, мы бы увидели это здесь, а не спрятали за высоким средним.

In [9]:
from sklearn.metrics import classification_report

pred = trainer.predict(test_ds)
preds = np.argmax(pred.predictions, axis=-1)

print(classification_report(pred.label_ids, preds, target_names=labels, digits=3))

                  precision    recall  f1-score   support

        Продукты      1.000     1.000     1.000        48
Кафе и рестораны      1.000     1.000     1.000        57
       Транспорт      1.000     1.000     1.000        49
          Аптека      1.000     1.000     1.000        35
     Развлечения      1.000     1.000     1.000        69
          Прочее      1.000     1.000     1.000        42

        accuracy                          1.000       300
       macro avg      1.000     1.000     1.000       300
    weighted avg      1.000     1.000     1.000       300



## Сохранение модели

Сохраняем лучшую модель и токенизатор в формате, который понимает инференс-обёртка
проекта. Кладём их в папку weights/categorizer. В инференсе категоризатор грузит ровно эту папку через
AutoModelForSequenceClassification и AutoTokenizer и работает в связке с правилами
по магазинам. Сохраняем и tokenizer тоже, потому что без него модель не сможет
превратить текст чека в токены на инференсе.

In [10]:
save_dir = str(root / "weights" / "categorizer")

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("модель и токенизатор сохранены в", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

модель и токенизатор сохранены в /content/Receipt-AI/weights/categorizer
